# 🎬 YouTube Shorts Automation — Kaggle Runtime

This notebook runs your **Telegram-controlled YouTube Shorts automation pipeline** on Kaggle.

### Setup (One-Time)
1. Go to **Sidebar → Secrets** (🔑 icon) and add your 8 API keys
2. Enable **Internet** (Settings → Internet → On)
3. Optional: Enable **GPU T4** for faster FFmpeg
4. Click **Run All**

In [ ]:
# ============================================
# CELL 1: Install System Dependencies
# ============================================
import subprocess
import sys

print("📦 Installing system packages...")
subprocess.run(["apt-get", "update", "-qq"], check=True,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run([
    "apt-get", "install", "-y", "-qq",
    "ffmpeg", "libsm6", "libxext6", "fonts-liberation", "git"
], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("📦 Installing Python packages...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "python-telegram-bot",
    "sqlalchemy[asyncio]",
    "asyncpg",
    "google-genai",
    "google-api-python-client",
    "google-auth-oauthlib",
    "google-auth-httplib2",
    "python-dotenv",
    "requests",
    "aiohttp",
    "aiofiles",
    "ffmpeg-python",
    "pillow",
    "pydantic",
    "pydantic-settings",
    "bcrypt",
    "apscheduler",
    "gTTS",
    "numpy",
    "httpx",
    "fastapi",
    "uvicorn"
], check=True)

print("✅ All dependencies installed!")

# Verify FFmpeg
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print(f"FFmpeg: {result.stdout.splitlines()[0]}")

# Show available resources
import psutil, shutil
ram_gb = psutil.virtual_memory().total / (1024**3)
disk = shutil.disk_usage('/')
print(f"\n🖥️ Available RAM: {ram_gb:.1f} GB")
print(f"🖥️ CPU cores: {psutil.cpu_count()}")
print(f"💾 Disk: {disk.free / (1024**3):.1f} GB free")

In [ ]:
# ============================================
# CELL 2: Load Secrets → Environment Variables
# ============================================
# 
# ⚠️ ADD THESE IN KAGGLE SIDEBAR → Secrets (🔑):
#   OPENROUTER_API_KEY
#   PEXELS_API_KEY
#   GEMINI_API_KEY
#   TELEGRAM_BOT_TOKEN
#   TELEGRAM_CHAT_ID
#   HF_TOKEN
#   POSTGRES_URL
#   GROQ_API_KEY
#
import os

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    
    SECRET_KEYS = [
        "OPENROUTER_API_KEY",
        "PEXELS_API_KEY",
        "GEMINI_API_KEY",
        "TELEGRAM_BOT_TOKEN",
        "TELEGRAM_CHAT_ID",
        "HF_TOKEN",
        "POSTGRES_URL",
        "GROQ_API_KEY",
    ]
    
    loaded = []
    missing = []
    
    for key in SECRET_KEYS:
        try:
            value = secrets.get_secret(key)
            if value:
                os.environ[key] = value
                loaded.append(key)
            else:
                missing.append(key)
        except Exception:
            missing.append(key)
    
    print(f"✅ Loaded {len(loaded)} secrets: {loaded}")
    if missing:
        print(f"⚠️ Missing secrets (add in sidebar → Secrets): {missing}")
        
except ImportError:
    print("⚠️ Not running on Kaggle — loading from .env file instead")
    from dotenv import load_dotenv
    load_dotenv()
    print("✅ Loaded from .env")

In [ ]:
# ============================================
# CELL 3: Clone Project Code from GitHub
# ============================================
import os
import subprocess
import shutil

WORK_DIR = "/kaggle/working/automation"
GITHUB_REPO = "https://github.com/Vinoth-46/YT-Automation.git"

# Clean previous run if needed
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    print(f"🗑️ Cleaned previous run at {WORK_DIR}")

# Clone from GitHub
print(f"📥 Cloning from GitHub: {GITHUB_REPO}")
result = subprocess.run(
    ["git", "clone", GITHUB_REPO, WORK_DIR],
    capture_output=True, text=True
)

if result.returncode != 0:
    print(f"❌ Git clone failed: {result.stderr}")
    raise RuntimeError("Could not clone repository. Check that the repo is public or use a token.")
else:
    print(f"✅ Cloned successfully to {WORK_DIR}")

# Create required writable directories
for d in ["outputs", "temp", "assets", "credentials", "assets/Watermark", "assets/cta_images"]:
    os.makedirs(os.path.join(WORK_DIR, d), exist_ok=True)

# Show what we got
contents = os.listdir(WORK_DIR)
print(f"\n📁 Working directory: {WORK_DIR}")
print(f"📁 Contents: {sorted(contents)}")

# Verify key modules exist
required = ["bot", "core", "engines", "utils"]
for module in required:
    path = os.path.join(WORK_DIR, module)
    if os.path.isdir(path):
        files = os.listdir(path)
        print(f"  ✅ {module}/ → {files}")
    else:
        print(f"  ❌ {module}/ NOT FOUND — code is incomplete!")

In [ ]:
# ============================================
# CELL 4: Set Python Path & Environment
# ============================================
import os
import sys

WORK_DIR = "/kaggle/working/automation"

# Add to Python path so 'from core.config import ...' works
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

os.chdir(WORK_DIR)
os.environ["PYTHONPATH"] = WORK_DIR
os.environ["PORT"] = "8080"

print(f"✅ sys.path[0] = {sys.path[0]}")
print(f"✅ cwd = {os.getcwd()}")
print(f"✅ PYTHONPATH = {os.environ['PYTHONPATH']}")

In [ ]:
# ============================================
# CELL 5: Sanity Check — Verify All Imports
# ============================================
import importlib

print("🔍 Testing imports...\n")
errors = []

# Test core.config
try:
    from core.config import settings
    print(f"  ✅ core.config")
    print(f"     OUTPUT_DIR: {settings.OUTPUT_DIR}")
    print(f"     TEMP_DIR:   {settings.TEMP_DIR}")
except Exception as e:
    print(f"  ❌ core.config: {e}")
    errors.append("core.config")

# Test engines
for mod in ["engines.script_engine", "engines.audio_engine", "engines.video_engine"]:
    try:
        importlib.import_module(mod)
        print(f"  ✅ {mod}")
    except Exception as e:
        print(f"  ❌ {mod}: {e}")
        errors.append(mod)

# Test bot
try:
    from bot.handlers import start_command, generate_command
    print(f"  ✅ bot.handlers")
except Exception as e:
    print(f"  ❌ bot.handlers: {e}")
    errors.append("bot.handlers")

# Test FFmpeg
import subprocess
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"  ✅ ffmpeg")
else:
    print(f"  ❌ ffmpeg not found")
    errors.append("ffmpeg")

# Final verdict
if errors:
    print(f"\n❌ {len(errors)} module(s) failed: {errors}")
    print("   Fix these before running Cell 6!")
else:
    print("\n✅ All checks passed! Ready to run bot.")

In [ ]:
# ============================================
# CELL 6: 🚀 START THE TELEGRAM BOT
# ============================================
# This cell runs FOREVER (until the 12h Kaggle limit)
# Your bot will respond to Telegram commands:
#   /start, /generate, /status, /schedule, etc.
#
# To stop: Interrupt the kernel (■ button)

import asyncio
import logging
import traceback
from telegram.ext import ApplicationBuilder, CommandHandler, CallbackQueryHandler
from telegram import Bot, BotCommand, MenuButtonCommands

from bot.handlers import (
    start_command, status_command, generate_command,
    schedule_command, view_schedule_command, cancel_command, button_callback
)
from core.config import settings

# Configure logging to show in notebook output
logging.basicConfig(
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    level=logging.INFO,
    force=True
)
logger = logging.getLogger("KaggleBot")


async def setup_bot_commands(bot):
    """Register bot commands."""
    commands = [
        BotCommand("start", "Start the bot and get welcome message"),
        BotCommand("generate", "Generate a new video now"),
        BotCommand("status", "Check recent job status"),
        BotCommand("schedule", "Set daily posting time (UTC)"),
        BotCommand("view_schedule", "View active schedules"),
        BotCommand("cancel", "Cancel current process")
    ]
    await bot.delete_my_commands()
    await bot.set_my_commands(commands)
    await bot.set_chat_menu_button(menu_button=MenuButtonCommands())
    logger.info("Bot commands registered")


async def init_services(application):
    """Initialize DB and scheduler."""
    try:
        from core.database import Database, init_db
        await init_db()
        logger.info("Database initialized")
    except Exception as e:
        logger.error(f"Database init failed (non-fatal): {e}")

    try:
        from core.scheduler import SchedulerService
        scheduler = SchedulerService()
        await scheduler.load_schedules()
        scheduler.start()
        application.bot_data["scheduler"] = scheduler
        logger.info("Scheduler started")
    except Exception as e:
        logger.error(f"Scheduler failed (non-fatal): {e}")


async def run_kaggle_bot():
    """Run the Telegram bot (Kaggle-optimized — no FastAPI needed)."""
    
    # Clear stale sessions
    logger.info("Clearing stale Telegram sessions...")
    try:
        temp_bot = Bot(token=settings.TELEGRAM_BOT_TOKEN)
        async with temp_bot:
            await temp_bot.delete_webhook(drop_pending_updates=True)
        logger.info("Stale sessions cleared")
    except Exception:
        pass

    await asyncio.sleep(3)

    application = (
        ApplicationBuilder()
        .token(settings.TELEGRAM_BOT_TOKEN)
        .pool_timeout(900.0)
        .connect_timeout(900.0)
        .read_timeout(900.0)
        .write_timeout(900.0)
        .build()
    )

    # Register handlers
    application.add_handler(CommandHandler('start', start_command))
    application.add_handler(CommandHandler('status', status_command))
    application.add_handler(CommandHandler('generate', generate_command))
    application.add_handler(CommandHandler('schedule', schedule_command))
    application.add_handler(CommandHandler('view_schedule', view_schedule_command))
    application.add_handler(CommandHandler('cancel', cancel_command))
    application.add_handler(CallbackQueryHandler(button_callback))

    logger.info("🚀 Bot starting on Kaggle...")

    async with application:
        await setup_bot_commands(application.bot)
        await init_services(application)

        await application.start()
        await application.updater.start_polling(
            drop_pending_updates=True,
            allowed_updates=["message", "callback_query"],
        )

        import psutil
        ram = psutil.virtual_memory()
        logger.info("=" * 50)
        logger.info("🟢 BOT IS LIVE! Send /start in Telegram")
        logger.info(f"🟢 RAM: {ram.total // (1024**3)}GB total, {ram.available // (1024**3)}GB free")
        logger.info("🟢 Kaggle session: ~12 hours runtime")
        logger.info("=" * 50)

        # Keep running until Kaggle kills the session
        try:
            while True:
                await asyncio.sleep(60)
        except (KeyboardInterrupt, asyncio.CancelledError):
            logger.info("Shutting down bot...")
        finally:
            await application.updater.stop()
            await application.stop()
            logger.info("Bot stopped cleanly.")


# Run the bot — handle Jupyter's already-running event loop
try:
    loop = asyncio.get_running_loop()
    # We're inside Jupyter/Kaggle which has a running loop
    await run_kaggle_bot()
except RuntimeError:
    # No event loop running (standalone script)
    asyncio.run(run_kaggle_bot())